# Human Activity Recognition using Hidden Markov Models
### Accelerometer + Gyroscope — Pixel 3a (Android) · Sensor Logger App
---
**Activities :** Still · Standing · Walking · Jumping  
**Sensors    :** Accelerometer (x, y, z) + Gyroscope (x, y, z) — merged CSV  
**Pipeline   :** Load → Resample → Window → Feature Extraction → HMM (Baum–Welch + Viterbi) → Evaluate

## 0 · Imports & Global Configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ML-Techniques-II-formative-2

Mounted at /content/drive
/content/drive/MyDrive/ML-Techniques-II-formative-2


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from itertools import permutations
from pathlib import Path
import warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi'        : 130,
    'font.size'         : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# ─── PATHS ────────────────────────────────────────────────────────────────────
TRAIN_DIR = Path('dataset/train')
TEST_DIR  = Path('dataset/test')
OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# ─── SAMPLING & WINDOWING ─────────────────────────────────────────────────────
TARGET_FS    = 50          # Hz  — harmonised rate for both devices
WINDOW_SEC   = 2.0         # seconds per window
OVERLAP      = 0.5         # 50 % overlap between consecutive windows
WINDOW_SAMP  = int(TARGET_FS * WINDOW_SEC)        # 100 samples
STEP_SAMP    = int(WINDOW_SAMP * (1 - OVERLAP))   #  50 samples

# ─── ACTIVITIES ───────────────────────────────────────────────────────────────
ACTIVITIES = ['still', 'standing', 'walking', 'jumping']
ACT2ID     = {a: i for i, a in enumerate(ACTIVITIES)}
ID2ACT     = {i: a for i, a in enumerate(ACTIVITIES)}
COLORS     = {'still':'#4e79a7','standing':'#f28e2b',
              'walking':'#59a14f','jumping':'#e15759'}

FEATURE_NAMES = [
    'RMS Accel',   'Var Accel',    'SMA',
    'Mean Gyro',   'Var Gyro',     'Corr ax-ay',
    'Dom Freq',    'Spec Energy',  'Spec Entropy',
]

print(f'Window : {WINDOW_SAMP} samples = {WINDOW_SEC} s  @  {TARGET_FS} Hz')
print(f'Step   : {STEP_SAMP} samples ({int(OVERLAP*100)}% overlap)')
print(f'Activities: {ACTIVITIES}')

Window : 100 samples = 2.0 s  @  50 Hz
Step   : 50 samples (50% overlap)
Activities: ['still', 'standing', 'walking', 'jumping']


## 1 · Data Collection Overview

Data were recorded with **Sensor Logger** and exported as merged CSVs  
(one file per recording, columns: `time · seconds_elapsed · accel_z/y/x · gyro_x/y/z`).

| Participant | Device | Native Sample Rate | Role |
|---|---|---|---|
| Member 1 | Pixel 3a (Android) | ~50 Hz | Training + Test |
| Member 2 | iPhone | ~50 Hz | Training + Test |

### File Naming Convention
```
dataset/train/
  still_1.csv  …  still_10.csv
  standing_1.csv  …  standing_10.csv
  walking_1.csv   …  walking_10.csv
  jumping_1.csv   …  jumping_10.csv

dataset/test/
  still_1.csv    # unseen test files
  standing_1.csv
  walking_1.csv
  jumping_1.csv
```

### Window Size Justification
At **50 Hz**, 1 sample = 20 ms.  
A **2-second window (100 samples)** was chosen because:
- It spans ≥ 1 full walking gait cycle (~0.9–1.2 s) and ≥ 1 full jump cycle.
- FFT resolution = 1/2 s = **0.5 Hz** — sufficient to separate still/standing  
  (near-DC) from walking (~1–2 Hz) and jumping (~2–3 Hz).
- Short enough to capture activity transitions for the HMM temporal model.

**50% overlap** doubles the number of windows and smooths feature transitions.

### Sampling Rate Harmonisation
Both devices export `seconds_elapsed` timestamps. Every recording is  
**linearly resampled to exactly 50 Hz** before any processing, ensuring  
all windows and frequency features are directly comparable.

## 2 · Data Loading & Resampling

In [21]:
# ─── Expected merged column aliases (handles minor naming variants) ────────────
COL_MAP = {
    # Accelerometer
    'accel_x':'ax', 'accel_y':'ay', 'accel_z':'az',
    'acceleration_x':'ax', 'acceleration_y':'ay', 'acceleration_z':'az',
    'x':'ax', 'y':'ay', 'z':'az',           # fallback for old single-sensor files
    # Gyroscope
    'gyro_x':'gx', 'gyro_y':'gy', 'gyro_z':'gz',
    'gyroscope_x':'gx', 'gyroscope_y':'gy', 'gyroscope_z':'gz',
}


def load_merged(path: Path) -> pd.DataFrame:
    """
    Load one merged CSV (accel + gyro in the same file),
    normalise column names, and resample to TARGET_FS via
    linear interpolation over seconds_elapsed.

    Expected columns (any capitalisation / separator):
        time  seconds_elapsed  accel_z  accel_y  accel_x  gyro_x  gyro_y  gyro_z
    """
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

    # Rename to canonical names
    df = df.rename(columns=COL_MAP)

    required = ['seconds_elapsed', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path.name}: missing columns {missing}. '
                         f'Found: {list(df.columns)}')

    df = df.sort_values('seconds_elapsed').dropna(subset=required).reset_index(drop=True)

    # ── Resample to TARGET_FS ──────────────────────────────────────────────────
    t_orig = df['seconds_elapsed'].values
    t_new  = np.arange(t_orig[0], t_orig[-1], 1.0 / TARGET_FS)

    resampled = {'seconds_elapsed': t_new}
    for col in ['ax', 'ay', 'az', 'gx', 'gy', 'gz']:
        resampled[col] = interp1d(
            t_orig, df[col].values, kind='linear',
            bounds_error=False, fill_value='extrapolate'
        )(t_new)

    return pd.DataFrame(resampled)


def discover_recordings(data_dir: Path):
    """
    Scan data_dir for CSVs following '{activity}_{n}.csv' pattern.
    Returns:
        files : {activity: [Path, ...]}
    """
    files = {a: [] for a in ACTIVITIES}

    for p in sorted(data_dir.glob('*.csv')):
        stem = p.stem.lower()
        # Remove common prefixes
        stem_clean = stem.removeprefix('test_').removeprefix('train_')
        act = next((a for a in ACTIVITIES if stem_clean.startswith(a)), None)
        if act is None:
            print(f'  [skip] {p.name}  — no matching activity in name')
            continue
        files[act].append(p)

    return files


# ── Load everything ────────────────────────────────────────────────────────────
train_files = discover_recordings(TRAIN_DIR)
test_files  = discover_recordings(TEST_DIR)

def load_group(file_dict):
    data = {a: [] for a in ACTIVITIES}
    for act, paths in file_dict.items():
        for p in paths:
            try:
                df = load_merged(p)
                dur = df['seconds_elapsed'].iloc[-1] - df['seconds_elapsed'].iloc[0]
                data[act].append(df)
                print(f'  ✓ {p.name:35s} → {len(df):4d} samples  ({dur:.2f} s)')
            except Exception as e:
                print(f'  ✗ {p.name}: {e}')
    return data

print('── Training files ───────────────────────────────────────')
train_data = load_group(train_files)

print('\n── Test files ───────────────────────────────────────────')
test_data  = load_group(test_files)

print('\n── Summary ──────────────────────────────────────────────')
for act in ACTIVITIES:
    n   = len(train_data[act])
    dur = sum((d['seconds_elapsed'].iloc[-1]-d['seconds_elapsed'].iloc[0])
              for d in train_data[act])
    nt  = len(test_data[act])
    print(f'  {act:10s}: {n:2d} train files  (~{dur:.0f} s)   |   {nt} test files')

  [skip] eliel_Jumping_eliel_01.csv.csv  — no matching activity in name
── Training files ───────────────────────────────────────
  ✓ still_eliel_01.csv.csv              →  500 samples  (9.98 s)
  ✓ still_eliel_02.csv.csv              →  479 samples  (9.56 s)
  ✓ still_eliel_03.csv.csv              →  492 samples  (9.82 s)
  ✓ still_eliel_04.csv.csv              →  542 samples  (10.82 s)
  ✓ still_eliel_05.csv.csv              →  513 samples  (10.24 s)
  ✓ still_eliel_06.csv.csv              →  494 samples  (9.86 s)
  ✓ still_merged_1.csv                  →  529 samples  (10.56 s)
  ✓ still_merged_2.csv                  →  530 samples  (10.58 s)
  ✓ still_merged_3.csv                  →  531 samples  (10.60 s)
  ✓ still_merged_4.csv                  →  524 samples  (10.46 s)
  ✓ still_merged_5.csv                  →  527 samples  (10.52 s)
  ✓ still_merged_6.csv                  →  527 samples  (10.52 s)
  ✓ standing_eliel_01.csv.csv           →  501 samples  (10.00 s)
  ✓ standing_eli

## 3 · Visualisation of Raw Sensor Signals

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle('Raw Sensor Signals — One Representative Recording per Activity',
             fontsize=14, fontweight='bold')

for row, act in enumerate(ACTIVITIES):
    samples = train_data[act] or test_data[act]
    if not samples:
        for col in range(2):
            axes[row, col].set_visible(False)
        continue
    df = samples[0]
    t  = df['seconds_elapsed'].values - df['seconds_elapsed'].values[0]
    c  = COLORS[act]

    ax_a = axes[row, 0]
    for axis_col, lbl in zip(['ax','ay','az'], ['x','y','z']):
        ax_a.plot(t, df[axis_col], alpha=0.85, lw=1, label=lbl)
    ax_a.set_title(f'{act.capitalize()} — Accelerometer (m/s²)', color=c, fontsize=11)
    ax_a.set_ylabel('m/s²')
    ax_a.legend(loc='upper right', fontsize=8, ncol=3)

    ax_g = axes[row, 1]
    for axis_col, lbl in zip(['gx','gy','gz'], ['x','y','z']):
        ax_g.plot(t, df[axis_col], alpha=0.85, lw=1, label=lbl)
    ax_g.set_title(f'{act.capitalize()} — Gyroscope (rad/s)', color=c, fontsize=11)
    ax_g.set_ylabel('rad/s')
    ax_g.legend(loc='upper right', fontsize=8, ncol=3)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_raw_signals.png', bbox_inches='tight')
plt.show()